[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PedroLormendez/jcclass/blob/main/notebooks/tutorial_era5_arco.ipynb)

# jcclass — Tutorial using ERA5 ARCO data

This notebook demonstrates how to use `jcclass` to compute Jenkinson-Collison Circulation Types from ERA5 mean sea-level pressure (MSLP) data accessed via the [ARCO-ERA5](https://github.com/google-research/arco-era5) dataset hosted on Google Cloud Storage.

---

## 1. Install dependencies

> **Note:** `gcsfs` and `zarr` are required to access ARCO-ERA5 data but are **not** bundled with `jcclass` — they are external dependencies you need to install separately.

In [9]:
# Install jcclass
!pip install jcclass

# Install ARCO-ERA5, ipywidgets access dependencies (external, not included in jcclass)
!pip install gcsfs zarr dask
!pip install ipywidgets

I0729 21:46:04.885004 19963107 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


I0729 21:46:05.613987 19963107 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


I0729 21:46:06.152017 19963107 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Load ERA5 MSLP data from ARCO-ERA5

ARCO-ERA5 is a public dataset hosted on Google Cloud Storage. No authentication is required for read access.

In [3]:
import xarray as xr

# Open the ARCO-ERA5 dataset (publicly accessible, no credentials needed)
ds = xr.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3',
    chunks={},
    consolidated=True,
    storage_options={'token': 'anon'},
)

# Select mean sea-level pressure for the Hurricane Sandy period (22–31 Oct 2012)
mslp = ds['mean_sea_level_pressure'].sel(time=slice('2012-10-30', '2012-10-31'))

# Thin the grid from 0.25° to 1° by selecting every 4th point
# The JCC algorithm uses fixed ±5° offsets to find neighbours, so any
# resolution works — fewer grid points just means faster computation
mslp = mslp.isel(latitude=slice(None, None, 4), longitude=slice(None, None, 4))

# Load into memory before computation
mslp = mslp.load()


I0729 21:30:44.780012 19965222 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0729 21:30:44.789954 19965244 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
I0729 21:30:45.893780 19963107 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0729 21:30:45.899574 19965273 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)


## 3. Compute Jenkinson-Collison Circulation Types

In [4]:
from jcclass import compute_cts

# Compute the 27 circulation types
cts = compute_cts(mslp)

100%|█████████████████████████████████████████████████████| 6/6 , Done ✓


## 4. Plot the results

Pick a timestamp from the dropdown and click **Plot** to render the circulation type map for that time step.

In [10]:
import ipywidgets as widgets
from IPython.display import display
from jcclass import plot_cts

# One dropdown entry per available timestamp in `cts`
time_options = [(str(t)[:19], t) for t in cts.time.values]

time_dropdown = widgets.Dropdown(
    options=time_options,
    description='Timestamp:',
    style={'description_width': 'initial'},
)
plot_button = widgets.Button(description='Plot', button_style='primary')
output = widgets.Output()

def on_plot_clicked(_):
    with output:
        output.clear_output(wait=True)
        plot_cts(cts.sel(time=time_dropdown.value), show=True)

plot_button.on_click(on_plot_clicked)
display(widgets.HBox([time_dropdown, plot_button]), output)

Output()